In [11]:
import numpy as np
import math

In [ ]:
def simulate_gbm(
    S0: float,
    sigma: float,
    T: float,
    n_steps: int,
    n_paths: int,
    r: float = 0.0,
    seed:int | None = None,
):
    """
    simulate geometric brownian motion under risk-neutral measure with r = 0

    parameters
    ----------
    S0 : float
        initial stock price

    sigma : float
        annualized volatility

    T : float
        time horizon in years

    n_steps : int
        number of simulation/hedging steps

    n_paths : int
        number of monte carlo paths

    r : float
        risk-free rate
        
    seed : int or None
        random seed for reproducibility
    
    returns
    -------
    paths : np.ndarray
        shape: (n_paths, n_steps + 1)
    
    """

    if S0 <= 0:
        raise ValueError("S0 must be positive")
    
    if sigma < 0:
        raise ValueError("sigma must be non-negative")
    
    if T <= 0:
        raise ValueError("T must be positive")
    
    if n_steps <= 0:
        raise ValueError("n_steps must be positive")
    
    if n_paths <= 0:
        raise ValueError("n_paths must be positive")

    dt = T / n_steps

    rng = np.random.default_rng(seed)

    # one indepenedent N(0, 1) shock for each path and timestamp
    Z = rng.standard_normal(size=(n_paths, n_steps))

    log_returns = (r -0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    log_paths = np.cumsum(log_returns, axis=1)

    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = S0
    paths[:, 1:] = S0 * np.exp(log_paths)

    return paths

In [5]:
# 30 trading days

S0 = 100.0
sigma = 0.20
T = 30 / 252
n_steps = 30
n_paths = 100_000

paths = simulate_gbm(
    S0=S0,
    sigma=sigma,
    T=T,
    n_steps=n_steps,
    n_paths=n_paths,
    seed=123
)

print(paths.shape)

(100000, 31)


In [ ]:
assert np.all(paths[:, 0] == S0)
assert np.all(paths > 0)

terminal_prices = paths[:, -1]
print(terminal_prices.mean())  # should be approx S0

100.02153466337425


In [9]:
theoretical_std = S0 * np.sqrt(np.exp(sigma**2 * T) - 1)

empirical_std = paths[:, -1].std()

print("theoretical:", theoretical_std)
print("simulated:  ", empirical_std)

theoretical: 6.908878815297882
simulated:   6.87632429753303


# simulate black scholes

In [ ]:
def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def black_scholes_call(
    S: float,
    K: float,
    sigma: float,
    T: float,
    t: float = 0.0,
    r: float = 0.0,
) -> float:
    """
    black-scholes price of a european call option

    parameters
    ----------
    S : float
        current stock price
        
    K : float
        strike price

    sigma : float
        annualized volatility

    T : float
        maturity time in years

    t : float
        current time in years
        
    r : float
        risk-free rate
    """

    if S <= 0:
        raise ValueError("S must be positive")
    
    if K <= 0:
        raise ValueError("K must be positive")

    if sigma <= 0:
        raise ValueError("sigma must be positive")

    tau = T - t

    if tau < 0:
        raise ValueError("t cannot be greater than T")

    if tau == 0:
        return max(S - K, 0.0)

    sqrt_tau = math.sqrt(tau)

    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * sqrt_tau)
    d2 = d1 - sigma * sqrt_tau

    call_price = S * norm_cdf(d1) - K * math.exp(-r * tau) * norm_cdf(d2)

    return call_price

    

In [13]:
def black_scholes_delta(
    S: float,
    K: float,
    sigma: float,
    T: float,
    t: float = 0.0,
    r: float = 0.0,
) -> float:
    """
    black-scholes delta of a european call option
    """

    if S <= 0:
        raise ValueError("S must be positive")
    
    if K <= 0:
        raise ValueError("K must be positive")

    if sigma <= 0:
        raise ValueError("sigma must be positive")

    tau = T - t

    if tau < 0:
        raise ValueError("t cannot be greater than T")

    if tau == 0:
        if S > K:
            return 1.0
        elif S < K:
            return 0.0
        else:
            return 0.5  # convention

    sqrt_tau = math.sqrt(tau)

    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * sqrt_tau)

    return norm_cdf(d1)

In [14]:
S = 100.0
K = 100.0
sigma = 0.20
T = 1.0
t = 0.0
r = 0.0


price = black_scholes_call(
    S=S,
    K=K,
    sigma=sigma,
    T=T,
    t=t,
    r=r
)

delta = black_scholes_delta(
    S=S,
    K=K,
    sigma=sigma,
    T=T,
    t=t,
    r=r  
)

print("call price:", price)
print("delta:     ", delta)

call price: 7.965567455405804
delta:      0.539827837277029


In [15]:
# finite difference delta test

epsilon = 1e-4

price_up = black_scholes_call(
    S=S + epsilon,
    K=K,
    sigma=sigma,
    T=T,
    t=t,
    r=r  
)

price_down = black_scholes_call(
    S=S - epsilon,
    K=K,
    sigma=sigma,
    T=T,
    t=t,
    r=r  
)

finite_difference_delta = (price_up - price_down) / (2.0 * epsilon)

print("analytical delta:       ", delta)
print("finite difference delta:", finite_difference_delta)

analytical delta:        0.539827837277029
finite difference delta: 0.5398278372581444


In [17]:
import numpy.typing as npt

def delta_hedge_path(
    stock_path: npt.NDArray[np.float64] | list[float],
    K: float,
    sigma: float,
    T: float,
    r: float = 0.0
):
    """
    delta hedge one short european call along one stock path

    parameters
    ----------
    stock_path:  array/list
        stock prices: [S_0, S_1, ..., S_N]
        
    K : float
        strike price
    
    sigma : float
        annualized volatility
    
    T : float
        maturity time in years
    
    r : float
        risk-free rate
    
    returns
    -------
    result : dict
        contains hedge positions, cash balances, terminal payoff, portfolio value, hedging P&L
    """

    stock_path = np.asarray(stock_path, dtype=float)

    if stock_path.ndim != 1:
        raise ValueError("stock_path must be one-dimensional")

    if len(stock_path) < 2:
        raise ValueError("stock_path must contain at least two prices")

    n_steps = len(stock_path) - 1

    dt = T / n_steps

    times = np.linspace(0.0, T, n_steps + 1)

    # price received for selling the call
    option_premium = black_scholes_call(
        S=stock_path[0],
        K=K,
        sigma=sigma,
        T=T,
        t=0.0,
        r=r
    )

    deltas = np.empty(n_steps)
    trades = np.empty(n_steps)
    cash_balances = np.empty(n_steps)

    cash = option_premium
    previous_delta = 0.0

    # hedge at t_0, ... t_{N - 1}
    # do not rebalance at expiration
    for i in range(n_steps):
        S = stock_path[i]
        t = times[i]

        # existing cash earns the risk-free rate
        if i > 0:
            cash *= math.exp(r * dt)

        new_delta = black_scholes_delta(
            S=S,
            K=K,
            sigma=sigma,
            T=T,
            t=t,
            r=r
        )

        # number of shares to buy/sell
        trade = new_delta - previous_delta

        cash -= trade * S

        deltas[i] = new_delta
        trades[i] = trade
        cash_balances[i] = cash

        previous_delta = new_delta

    # cash accrues over final interval
    cash *= math.exp(r * dt)

    S_T = stock_path[-1]
    stock_value = previous_delta * S_T

    terminal_portfolio = stock_value + cash

    payoff = max(S_T - K, 0.0)
    hedging_pnl = terminal_portfolio - payoff

    return {
        "times": times,
        "stock_path": stock_path,
        "option_premium": option_premium,
        "deltas": deltas,
        "trades": trades,
        "cash_balances": cash_balances,
        "terminal_portfolio": terminal_portfolio,
        "payoff": payoff,
        "hedging_pnl": hedging_pnl
    }

In [22]:
stock_path = [100, 95, 90, 100, 105]
K = 100
sigma = 0.20
T = 1.0

result = delta_hedge_path(
    stock_path=stock_path,
    K=K,
    sigma=sigma,
    T=T
)

initial_portfolio = result["deltas"][0] * stock_path[0] + result["cash_balances"][0]

print(initial_portfolio)
print(result["option_premium"])

7.965567455405804
7.965567455405804


In [26]:
K = 100
sigma = 0.20
T = 30 / 252
n_steps = 30
r = 0.0

paths = simulate_gbm(
    S0=100,
    sigma=sigma,
    T=T,
    n_steps=n_steps,
    n_paths=1,
    seed=123
)

stock_path = paths[0]

result = delta_hedge_path(
    stock_path=stock_path,
    K=K,
    sigma=sigma,
    T=T,
    r=r 
)

print('stock path:')
print(result["stock_path"])

print("\ndeltas:")
print(result["deltas"])

print("\ntrades:")
print(result["trades"])

print("\ncash:")
print(result["cash_balances"])

print("\nterminal option payoff:")
print(result["payoff"])

print("\nterminal hedge portfolio:")
print(result["terminal_portfolio"])

stock path:
[100.          98.75371894  98.28938366  99.88934166 100.12580775
 101.28536483 102.01637979 101.19358174 101.87880539 101.46519482
 101.0458875  101.16163391  99.22750979 100.72115008  99.86522311
 101.12368313 101.28947158 103.25533593 102.39221784 101.98269082
 102.40947391  99.59264736 100.6289314  102.59438017 104.05298319
 105.0388269  104.83750166 106.53596557 107.978786   108.50562061
 108.50400055]

deltas:
[0.51376209 0.44001876 0.41080328 0.506311   0.52061295 0.5926015
 0.63844352 0.58962594 0.63477755 0.6105702  0.58427194 0.59396826
 0.45289333 0.56525296 0.49937591 0.60002035 0.61614051 0.76667015
 0.71345083 0.68819561 0.73155498 0.46450231 0.57681737 0.78378957
 0.90368371 0.96071875 0.97044889 0.99820583 0.9999921  1.        ]

trades:
[ 5.13762086e-01 -7.37433227e-02 -2.92154826e-02  9.55077169e-02
  1.43019574e-02  7.19885472e-02  4.58420210e-02 -4.88175814e-02
  4.51516089e-02 -2.42073489e-02 -2.62982599e-02  9.69631748e-03
 -1.41074932e-01  1.12359637e